# Ex-Situ Quick Pipeline

Use this notebook for flat, legacy, or version-2 ex-situ NCI data. Beam history, OB, and BG inputs are independent and optional. Version-2 beam correction automatically prefers `metadata/<experiment>/<run>/tpx3_exposure_manifest.json`, so it still works after raw TPX3 files are archived.

Edit the paths in the next cell, then run the notebook from top to bottom.

In [ ]:
from pathlib import Path

# Required: folder containing the sample TIFF images.
# This can be a flat TIFF folder, or run folders with TIFFs directly inside
# each run or inside each run's final/ folder. Version-2 example:
# /proposal/data/experiments/final/<experiment>
data_dir = Path(r"/path/to/sample/data")

# Optional: set to None or "" to skip beam-history intensity correction.
beamhistory_path = None  # Path(r"/path/to/beamhistory.csv")

# Optional: set to None or "" to skip OB correction.
# This can be one OB TIFF file or a folder containing OB TIFFs.
ob_path = None  # Path(r"/path/to/ob")

# Optional: set to None or "" to skip background correction.
# An external BG receives the same available beam/OB corrections.
bg_path = None  # Path(r"/path/to/background")

# Optional: default is data_dir.parent / f"results_{data_dir.name}".
results_dir = None

# Optional: default average filename is data_dir.parent.name + ".tif".
sample_name = None

# Processing settings. tpx_time is only the legacy raw-mtime fallback;
# a version-2 exposure manifest supplies the actual per-file durations.
tpx_time = 2
n_tpx_per_image = 150
ob_bin_y = 2
ob_bin_x = 2
final_subfolder = "final"
file_filter = None

# Optional: background correction after OBINT correction.
background_correction = False
background_n_images = 10
clip_negatives = False

data_dir

In [ ]:
import sys
from pathlib import Path

TOOLBOX_DIR = Path(r"/data01/software/toolbox")
if str(TOOLBOX_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLBOX_DIR))

from exsitu import run_exsitu_pipeline

print(f"Using toolbox: {TOOLBOX_DIR}")

In [ ]:
result = run_exsitu_pipeline(
    data_dir=data_dir,
    beamhistory_path=beamhistory_path,
    ob_path=ob_path,
    bg_path=bg_path,
    results_dir=results_dir,
    sample_name=sample_name,
    tpx_time=tpx_time,
    n_tpx_per_image=n_tpx_per_image,
    ob_bin_y=ob_bin_y,
    ob_bin_x=ob_bin_x,
    final_subfolder=final_subfolder,
    file_filter=file_filter,
    background_correction=background_correction,
    background_n_images=background_n_images,
    clip_negatives=clip_negatives,
)

result

In [ ]:
for key, value in result.items():
    print(f"{key:24s}: {value}")

## Preview Average Image

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import tifffile

average_paths = result.get("average_paths") or {}
if average_paths:
    preview_paths = [Path(path) for path in average_paths.values()]
else:
    preview_paths = [Path(result["average_path"])]

preview_paths = [path for path in preview_paths if path is not None and path.is_file()]
if not preview_paths:
    print("No average images found yet.")
else:
    fig, axes = plt.subplots(1, len(preview_paths), figsize=(6 * len(preview_paths), 5.5), squeeze=False)
    for ax, path in zip(axes[0], preview_paths):
        avg = tifffile.imread(path)
        vmin, vmax = np.nanpercentile(avg, [1, 99])
        im = ax.imshow(avg, cmap="gray", vmin=vmin, vmax=vmax)
        ax.set_title(path.name)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
plt.show()

## Preview Corrected Stack

In [ ]:
corrected_dir = Path(result.get("final_dir") or result["obint_corrected_dir"])
corrected_files = sorted(
    p for p in corrected_dir.iterdir()
    if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}
)
preview_files = corrected_files[: min(6, len(corrected_files))]

if not preview_files:
    print(f"No corrected TIFF files found in {corrected_dir}")
else:
    fig, axes = plt.subplots(1, len(preview_files), figsize=(3 * len(preview_files), 3), squeeze=False)
    for ax, path in zip(axes[0], preview_files):
        img = tifffile.imread(path)
        vmin, vmax = np.nanpercentile(img, [1, 99])
        ax.imshow(img, cmap="gray", vmin=vmin, vmax=vmax)
        ax.set_title(path.name, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## Notes

- `results_<data_dir folder name>/uncorrected/` contains copied raw TIFFs.
- `results_<data_dir folder name>/obint_corrected/` contains the OBINT-corrected TIFF stack.
- `background.tif` and `final/` are written when `bg_path` is supplied or legacy `background_correction = True`.
- The average TIFF is named after the parent folder of `data_dir` unless `sample_name` is set.
- If no beam-history path is supplied, intensity correction is skipped.
- If no OB path is supplied, OB correction is skipped.
- If no BG path is supplied, background correction is skipped unless the legacy sample-frame BG checkbox is enabled.